In [26]:
import numpy as np
import pandas as pd
import pymc as pm
import pgmpy
import arviz as az
import warnings
warnings.filterwarnings("ignore")

In [27]:
df = pd.read_csv('/Users/ziqianzhao/Desktop/2026 Winter/Bayesian Analysis/Project/Bayesian-ML-with-Gen-AI-Final-Project/Dataset/loan.csv')

In [28]:
# To keep this project clean and straight forward, we only consider individual borrowers and filter out joint loans 
df_1 = df[df["application_type"] == "Individual"].copy()

In [29]:
df_1.shape

(2139958, 145)

In [30]:
# Identify joint-related columns
joint_cols = [col for col in df.columns 
              if "joint" in col.lower() or col.startswith("sec_app_")]

# Drop them
df_1 = df_1.drop(columns=joint_cols)

In [31]:
df_1.shape

(2139958, 131)

In [32]:
print(df_1.columns.tolist())

['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m', 'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'il_util', 'open

In [33]:
# This core candidate are the most influential predictors (in a economic context not statistical context) that we should alwyas keep
core_hist = [
    'delinq_2yrs',
    'num_accts_ever_120_pd',
    'num_tl_90g_dpd_24m',
    'mths_since_last_delinq',
    'pub_rec_bankruptcies',
    'tax_liens',
    'mths_since_last_major_derog',
    'pct_tl_nvr_dlq'
]
core_repay = [
    'dti',
    'revol_util',
    'percent_bc_gt_75',
    'bc_util',
    'all_util',
    'installment',
    'loan_amnt',
    'annual_inc',
    'avg_cur_bal',
    'total_bal_ex_mort',
    'tot_cur_bal'
]
core_stability = [
    'earliest_cr_line',
    'mo_sin_old_rev_tl_op',
    'total_acc',
    'open_acc'
]
core_riskbehavior = [
    'inq_last_6mths',
    'inq_last_12m',
    'num_tl_op_past_12m',
    'acc_now_delinq',
    'chargeoff_within_12_mths',
    'collections_12_mths_ex_med'
]
	

In [34]:
# Secondary cancidates
sec_candidate = [
    'grade',
    'sub_grade',
    'int_rate',
    'term',
    'purpose',
    'initial_list_status',
    'num_actv_bc_tl',
    'num_rev_accts',
    'num_il_tl',
    'total_rev_hi_lim',
    'total_bc_limit',
    'total_il_high_credit_limit',
    'max_bal_bc',
    'home_ownership',
    'addr_state',
    'zip_code',
    'emp_length',
    'verification_status'
]

In [35]:
# These variablea are identifier for records and/or date, which are not considered as useful and will be deleted
del_col = [
    'id',
    'member_id',
    'url',
    'issue_d',
    'last_credit_pull_d',
    'desc',
    'title',
    'emp_title',
    'funded_amnt',
    'funded_amnt_inv',
    'last_pymnt_d'
]

In [36]:
df_clean = df_1.drop(columns=del_col)

In [37]:
df_clean.shape

(2139958, 120)

In [38]:
df_clean = df_clean.dropna(axis=1, how='all')
df_clean.shape

(2139958, 120)

In [39]:
# Drop hardship and settlement columns which are post-loan stage, not applicable for our preidiction
cols_to_drop = [col for col in df.columns 
                if col.startswith("hardship") 
                or col.startswith("settlement")]
cols_to_drop = cols_to_drop + ['debt_settlement_flag', 'debt_settlement_flag_date', 'deferral_term', 
                               'payment_plan_start_date', 'orig_projected_additional_accrued_interest','next_pymnt_d']
df_clean = df_clean.drop(columns=cols_to_drop)

In [40]:
df_clean.shape

(2139958, 97)

In [41]:
missing_ratio = df_clean.isna().mean()
high_missing = missing_ratio[missing_ratio > 0.5]

In [42]:
high_missing.index

Index(['mths_since_last_delinq', 'mths_since_last_record',
       'mths_since_last_major_derog', 'mths_since_recent_bc_dlq',
       'mths_since_recent_revol_delinq'],
      dtype='object')

For variables measuring the number of months since a negative credit event (e.g., delinquency or derogatory record), missing values typically indicate that the event has never occurred rather than that the information is unknown. Because “never occurred” is fundamentally different from “occurred recently,” we explicitly encode this distinction. 

We first create an indicator variable (e.g., has_delinq) to capture whether the borrower has ever experienced the event. We then replace missing values with a large constant (e.g., 999) to represent a very long time since the event, effectively approximating “no history of such behavior.” This approach preserves the economic meaning of the variable, avoids incorrectly interpreting missing values as zero (which would imply a very recent event), and allows the model to distinguish between borrowers with recent adverse events and those with no such history.

In [43]:
df_clean['has_delinq'] = df_clean['mths_since_last_delinq'].notna().astype(int)
df_clean['mths_since_last_delinq'] = df_clean['mths_since_last_delinq'].fillna(999)
df_clean['mths_since_last_record'] = df_clean['mths_since_last_record'].fillna(999)
df_clean['mths_since_last_major_derog'] = df_clean['mths_since_last_major_derog'].fillna(999)
df_clean['mths_since_recent_bc_dlq'] = df_clean['mths_since_recent_bc_dlq'].fillna(999)
df_clean['mths_since_recent_revol_delinq'] = df_clean['mths_since_recent_revol_delinq'].fillna(999)

In [44]:
df_clean.shape

(2139958, 98)

In [45]:
df_clean['loan_status'].unique()

array(['Current', 'Fully Paid', 'Late (31-120 days)', 'In Grace Period',
       'Charged Off', 'Late (16-30 days)', 'Default',
       'Does not meet the credit policy. Status:Fully Paid',
       'Does not meet the credit policy. Status:Charged Off'],
      dtype=object)

In [46]:
df_1['loan_status'].unique()

array(['Current', 'Fully Paid', 'Late (31-120 days)', 'In Grace Period',
       'Charged Off', 'Late (16-30 days)', 'Default',
       'Does not meet the credit policy. Status:Fully Paid',
       'Does not meet the credit policy. Status:Charged Off'],
      dtype=object)

In [47]:
# mapping the status to binary: default and non-default
loan_status_mapping = {
    'Fully Paid':0,
    'Current': 0,
    'In Grace Period': 0,
    'Late (16-30 days)': 1,
    'Late (31-120 days)': 1,
    'Charged Off': 1,
    'Default': 1,
    'Does not meet the credit policy. Status:Fully Paid': 0,
    'Does not meet the credit policy. Status:Charged Off':1
}

# Apply the mapping to the 'Loan_Status' column
df_clean['loan_status_binary'] = df_clean['loan_status'].map(loan_status_mapping)
df_clean.drop('loan_status', axis=1, inplace=True)

df_clean['loan_status_binary'].value_counts()

loan_status_binary
0    1860208
1     279750
Name: count, dtype: int64

In [48]:
# Random sample 50k data from this dataset for model training
df_clean = df_clean.dropna(subset=['loan_status_binary'])
df_clean['loan_status_binary'].value_counts(normalize=True)
df_sample = df_clean.sample(
    n=50000,
    random_state=42
)

In [49]:
print("Full data ratio:")
print(df_clean['loan_status_binary'].value_counts(normalize=True))

print("\nSample ratio:")
print(df_sample['loan_status_binary'].value_counts(normalize=True))

Full data ratio:
loan_status_binary
0    0.869273
1    0.130727
Name: proportion, dtype: float64

Sample ratio:
loan_status_binary
0    0.86958
1    0.13042
Name: proportion, dtype: float64


In [50]:
# Save data for further part
df_sample.to_csv("lendingclub_sample_50k.csv", index=False)